### B1. Inspecting CSV files

In [0]:
LIST '/Volumes/dbacademy_ecommerce/v01/raw/sales-csv'

In [0]:
SELECT * 
FROM csv.`/Volumes/dbacademy_ecommerce/v01/raw/sales-csv`
LIMIT 5;

In [0]:
SELECT * 
FROM read_files(
        "/Volumes/dbacademy_ecommerce/v01/raw/sales-csv",
        format => "csv"
      )
LIMIT 5;

### B2. Using CSV Options with `read_files()`

In [0]:
SELECT * 
FROM read_files(
        "/Volumes/dbacademy_ecommerce/v01/raw/sales-csv",
        format => "csv",
        sep => "|",
        header => true
      )
LIMIT 5;

In [0]:
-- Drop the table if it exists for demonstration purposes
DROP TABLE IF EXISTS sales_bronze;


-- Create the Delta table
CREATE TABLE sales_bronze AS
SELECT 
  *,
  _metadata.file_modification_time AS file_modification_time,
  _metadata.file_name AS source_file, 
  current_timestamp() as ingestion_time 
FROM read_files(
        "/Volumes/dbacademy_ecommerce/v01/raw/sales-csv",
        format => "csv",
        sep => "|",
        header => true
      );


-- Display the table
SELECT *
FROM sales_bronze

### B3. (BONUS) Python Equivalent

In [0]:
%python

df = (spark
      .read 
      .option("header", True) 
      .option("sep","|") 
      .option("rescuedDataColumn", "_rescued_data")       # <--------- Add the rescued data column
      .csv("/Volumes/dbacademy_ecommerce/v01/raw/sales-csv")
    )

df.display()

Define a schema with read_files, to handle schema missmatch in rescued data column

In [0]:
SELECT *
FROM read_files(
        DA.paths_working_dir || '/csv_demo_files/malformed_example_1_data.csv',
        format => "csv",
        sep => "|",
        header => true,
        schema => '''
            order_id INT, 
            email STRING, 
            transactions_timestamp BIGINT''', 
        rescueddatacolumn => '_rescued_data'    -- Create the _rescued_data column
      );

### C2. Handling Missing Headers During Ingestion 

In [0]:
-- Drop the table if it exists for demonstration purposes
DROP TABLE IF EXISTS demo_4_example_2_bronze;

-- Create Delta table by ingesting CSV file
CREATE OR REPLACE TABLE demo_4_example_2_bronze AS
SELECT *
FROM read_files(
        DA.paths_working_dir || '/csv_demo_files/malformed_example_2_data.csv',
        format => "csv",
        sep => "|",
        header => true
      );


-- Display the table
SELECT *
FROM demo_4_example_2_bronze;

In [0]:
SELECT
  cast(_rescued_data:_c0 AS BIGINT) AS order_id,
  *
FROM read_files(
        DA.paths_working_dir || '/csv_demo_files/malformed_example_2_data.csv',
        format => "csv",
        sep => "|",
        header => true
      )